# V1-13: LLM 구조화 검색어 제안이 기업 후보 탐색을 개선하는가

이 평가는 로컬 `qwen3.6:27b`가 줄임말·오타·표기 차이에서 OpenDART 검색어 후보를 제안했을 때, 기존의 정규화+문자열 유사도 후보 생성보다 정답 기업을 더 잘 후보 집합에 넣는지 확인한다.

> 이 notebook은 최종 Agent, Tool 호출, OpenDART 재무 조회를 구현하지 않는다. LLM은 `corp_code`를 선택하지 않으며, 구조화된 **기업명 검색어**만 반환한다.

## 1. 비교 대상과 판정 범위

- **A. 기존 기준선**: 사용자 입력 기업명으로 정규화 exact 후보와 문자열 유사(fuzzy) 후보를 함께 만든다.
- **B. LLM 검색어 확장**: A의 후보를 만든 뒤, *고유 exact 후보가 없는 경우에만* LLM이 최대 2개의 검색어를 제안한다. 각 제안도 같은 OpenDART 인덱스에서 다시 탐색하고 `corp_code` 기준으로 병합한다.

정식 법인명이 OpenDART에서 고유 exact로 확인되면 LLM은 호출하지 않는다. `삼전`처럼 원문 exact 후보가 여러 개인 경우에는 LLM이 삼성전자를 제안하더라도 원문 동명 후보를 유지한다. 이번 평가는 후보 회수와 제안 품질만 다루므로 LLM 제안으로 자동 확정하지 않는다.

### 이번에 보는 지표

- **Candidate Recall**: 정답 기업이 후보 집합에 포함됐는가
- **LLM Expansion Contribution**: 기준선이 놓친 정답을 LLM 확장이 새로 포함했는가
- **LLM Search-term Validity**: LLM 제안 검색어가 실제 OpenDART 후보를 하나 이상 만들었는가
- **Structured-output Success Rate**: LLM 응답이 정해 둔 Pydantic 스키마를 통과했는가
- **LLM Call Rate / 지연시간 / 후보 수**: 후보 회수 이득에 비해 호출 비용과 UX 부담이 감당 가능한가

지표 정의는 `docs/EVALUATION_METRICS.md`에 기록했다. 이번 1회 실행은 로컬 모델의 후보 가능성을 확인하는 개발 평가다. 프롬프트·모델·자동 확정 정책을 최종 채택하려면 이후 독립 holdout 평가가 필요하다.

In [ ]:
import io
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from math import ceil
import os
from pathlib import Path
import re
from statistics import median
import sys
import time
from typing import Literal
import xml.etree.ElementTree as ET
import zipfile

from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from rapidfuzz import fuzz, process

In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
MODEL_NAME = os.environ.get('OLLAMA_MODEL', 'qwen3.6:27b')
TEMPERATURE = 0
PROMPT_VERSION = 'entity-search-expansion-v1'

print(f'프로젝트 루트: {project_root}')
print(f'Ollama 주소: {OLLAMA_BASE_URL}')
print(f'모델: {MODEL_NAME} / temperature={TEMPERATURE}')

In [ ]:
sys.path.insert(0, str(project_root / 'src'))

from dart.client import CORP_CODE_URL, _get_bytes


def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    zip_stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(zip_stream):
        error_text = zip_bytes.decode('utf-8', errors='replace')
        raise RuntimeError(f'기업코드 응답이 ZIP이 아닙니다: {error_text}')
    with zipfile.ZipFile(zip_stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])

    root = ET.fromstring(xml_bytes)
    return [
        {child.tag: child.text or '' for child in item}
        for item in root.findall('list')
    ]


corp_records = load_corp_records(api_key)
retrieved_at = datetime.now().astimezone().isoformat(timespec='seconds')
print(f'OpenDART 기업코드 조회 시각: {retrieved_at}')
print(f'전체 법인 레코드 수: {len(corp_records):,}')

In [ ]:
def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())


def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(by_code.values()) for key, by_code in index.items()}


name_index = build_name_index(corp_records)
name_keys = list(name_index)

TARGET_COMPANIES = {
    '삼성전자': ('삼성전자', '005930'),
    'SK하이닉스': ('SK하이닉스', '000660'),
    'LG전자': ('LG전자', '066570'),
    '현대자동차': ('현대자동차', '005380'),
    'NAVER': ('NAVER', '035420'),
    '카카오': ('카카오', '035720'),
    'POSCO홀딩스': ('POSCO홀딩스', '005490'),
    '삼성바이오로직스': ('삼성바이오로직스', '207940'),
    '셀트리온': ('셀트리온', '068270'),
    '기아': ('기아', '000270'),
}

target_records = {}
for key, (official_name, stock_code) in TARGET_COMPANIES.items():
    matches = [
        record for record in corp_records
        if normalize_name(record['corp_name']) == normalize_name(official_name)
        and record['stock_code'].strip() == stock_code
    ]
    if len(matches) != 1:
        raise AssertionError(f'{key}: OpenDART 기준 법인 검증 실패 ({len(matches)}건)')
    target_records[key] = matches[0]

print(f'정규화 이름 키 수: {len(name_keys):,}')
for key, record in target_records.items():
    print(f'{key}: {record["corp_name"]} / {record["corp_code"]} / {record["stock_code"]}')

## 2. 평가 입력과 정답

`원문 기업 표현`은 이후 실제 Router가 추출할 입력을 간단히 고정한 것이다. 이번 노트북은 **문장에서 기업명을 추출하는 성능**이 아니라, 이미 추출한 원문 표현을 LLM이 더 나은 OpenDART 검색어로 확장하는 성능을 본다.

`삼전`은 삼성전자를 의도한 질문이지만 실제 OpenDART에 동명 법인이 있으므로, 정답 후보 포함과 별개로 최종 상태는 `NEEDS_CONFIRMATION`이어야 한다. 존재하지 않는 입력은 LLM이 그럴듯한 실제 기업을 억지로 제안하는지 확인한다.

In [ ]:
TEST_CASES = [
    {'질문': '삼성전자의 2024년 매출을 알려줘.', '원문 기업 표현': '삼성전자', '유형': '정식 명칭', '기대 대상': ['삼성전자'], '기대 상태': 'RESOLVED'},
    {'질문': '삼성 전자 실적을 요약해줘.', '원문 기업 표현': '삼성 전자', '유형': '띄어쓰기', '기대 대상': ['삼성전자'], '기대 상태': 'RESOLVED'},
    {'질문': '삼전의 최근 매출 추세를 보여줘.', '원문 기업 표현': '삼전', '유형': '시장 약칭·동명', '기대 대상': ['삼성전자'], '기대 상태': 'NEEDS_CONFIRMATION'},
    {'질문': '삼성전쟈 영업이익을 알려줘.', '원문 기업 표현': '삼성전쟈', '유형': '한 글자 오타', '기대 대상': ['삼성전자'], '기대 상태': 'INTERPRETATION_CANDIDATE'},
    {'질문': 'SK하이닉스의 사업보고서 기준 매출은?', '원문 기업 표현': 'SK하이닉스', '유형': '정식 명칭', '기대 대상': ['SK하이닉스'], '기대 상태': 'RESOLVED'},
    {'질문': 'sk 하이닉스 실적을 알려줘.', '원문 기업 표현': 'sk 하이닉스', '유형': '영문 대소문자·띄어쓰기', '기대 대상': ['SK하이닉스'], '기대 상태': 'RESOLVED'},
    {'질문': '하이닉스 매출 추세를 보고 싶어.', '원문 기업 표현': '하이닉스', '유형': '부분 표현', '기대 대상': ['SK하이닉스'], '기대 상태': 'INTERPRETATION_CANDIDATE'},
    {'질문': '하닉 실적을 정리해줘.', '원문 기업 표현': '하닉', '유형': '시장 약칭', '기대 대상': ['SK하이닉스'], '기대 상태': 'INTERPRETATION_CANDIDATE'},
    {'질문': 'LG전자 매출을 알려줘.', '원문 기업 표현': 'LG전자', '유형': '정식 명칭', '기대 대상': ['LG전자'], '기대 상태': 'RESOLVED'},
    {'질문': '엘지전자 영업이익 추이를 알려줘.', '원문 기업 표현': '엘지전자', '유형': '한글 표기', '기대 대상': ['LG전자'], '기대 상태': 'INTERPRETATION_CANDIDATE'},
    {'질문': '현대자동차의 최근 실적은?', '원문 기업 표현': '현대자동차', '유형': '정식 명칭', '기대 대상': ['현대자동차'], '기대 상태': 'RESOLVED'},
    {'질문': '현대차 매출을 알려줘.', '원문 기업 표현': '현대차', '유형': '시장 약칭', '기대 대상': ['현대자동차'], '기대 상태': 'INTERPRETATION_CANDIDATE'},
    {'질문': '현차 최근 매출 추세는?', '원문 기업 표현': '현차', '유형': '시장 약칭', '기대 대상': ['현대자동차'], '기대 상태': 'INTERPRETATION_CANDIDATE'},
    {'질문': 'NAVER의 매출을 알려줘.', '원문 기업 표현': 'NAVER', '유형': '정식 영문 명칭', '기대 대상': ['NAVER'], '기대 상태': 'RESOLVED'},
    {'질문': '네이버 영업이익을 알려줘.', '원문 기업 표현': '네이버', '유형': '한글 표기', '기대 대상': ['NAVER'], '기대 상태': 'INTERPRETATION_CANDIDATE'},
    {'질문': '카카오의 매출을 알려줘.', '원문 기업 표현': '카카오', '유형': '정식 명칭', '기대 대상': ['카카오'], '기대 상태': 'RESOLVED'},
    {'질문': 'POSCO홀딩스 실적을 분석해줘.', '원문 기업 표현': 'POSCO홀딩스', '유형': '정식 영문 혼합 명칭', '기대 대상': ['POSCO홀딩스'], '기대 상태': 'RESOLVED'},
    {'질문': '포스코홀딩스 영업이익을 보여줘.', '원문 기업 표현': '포스코홀딩스', '유형': '한글 표기', '기대 대상': ['POSCO홀딩스'], '기대 상태': 'INTERPRETATION_CANDIDATE'},
    {'질문': '삼바의 최근 실적은?', '원문 기업 표현': '삼바', '유형': '시장 약칭', '기대 대상': ['삼성바이오로직스'], '기대 상태': 'INTERPRETATION_CANDIDATE'},
    {'질문': '셀트 매출 추세를 알려줘.', '원문 기업 표현': '셀트', '유형': '시장 약칭', '기대 대상': ['셀트리온'], '기대 상태': 'INTERPRETATION_CANDIDATE'},
    {'질문': '기아차의 영업이익을 알려줘.', '원문 기업 표현': '기아차', '유형': '시장 약칭', '기대 대상': ['기아'], '기대 상태': 'INTERPRETATION_CANDIDATE'},
    {'질문': '엘지 계열사의 실적을 알려줘.', '원문 기업 표현': '엘지', '유형': '포괄 표현', '기대 대상': [], '기대 상태': 'NEEDS_CONFIRMATION'},
    {'질문': '현대 실적을 알려줘.', '원문 기업 표현': '현대', '유형': '포괄 표현', '기대 대상': [], '기대 상태': 'NEEDS_CONFIRMATION'},
    {'질문': '감자우주물산2026의 매출을 알려줘.', '원문 기업 표현': '감자우주물산2026', '유형': '대상 부재', '기대 대상': [], '기대 상태': 'NOT_FOUND'},
    {'질문': 'BLUEORBITVENTURES2049의 실적을 알려줘.', '원문 기업 표현': 'BLUEORBITVENTURES2049', '유형': '대상 부재', '기대 대상': [], '기대 상태': 'NOT_FOUND'},
    {'질문': 'QXZ테스트법인9999 재무정보를 알려줘.', '원문 기업 표현': 'QXZ테스트법인9999', '유형': '대상 부재', '기대 대상': [], '기대 상태': 'NOT_FOUND'},
]

for case in TEST_CASES:
    if case['유형'] == '대상 부재':
        assert normalize_name(case['원문 기업 표현']) not in name_index

print(f'전체 사례: {len(TEST_CASES)}개')
print(f'정답 기업이 있는 사례: {sum(bool(case["기대 대상"]) for case in TEST_CASES)}개')
print(f'LLM을 호출하지 않아야 하는 정식·고유 exact 사례는 다음 셀에서 실제로 확인합니다.')

In [ ]:
FUZZY_KEY_LIMIT = 50
FUZZY_RESULT_LIMIT = 5


def exact_records(query):
    return name_index.get(normalize_name(query), [])


def fuzzy_candidate_records(query):
    scored_keys = process.extract(
        normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT
    )
    best_by_code = {}
    for key, score, _ in scored_keys:
        for record in name_index[key]:
            current = best_by_code.get(record['corp_code'])
            if current is None or score > current[1]:
                best_by_code[record['corp_code']] = (record, score)
    return sorted(
        best_by_code.values(),
        key=lambda item: (item[1], bool(item[0]['stock_code'].strip()), item[0]['modify_date']),
        reverse=True,
    )[:FUZZY_RESULT_LIMIT]


def add_candidate(candidates, record, source, fuzzy_score=None):
    current = candidates.setdefault(
        record['corp_code'],
        {'record': record, 'sources': set(), 'fuzzy_score': None},
    )
    current['sources'].add(source)
    if fuzzy_score is not None:
        current['fuzzy_score'] = max(current['fuzzy_score'] or 0, fuzzy_score)


def union_candidates(query, source_prefix='raw'):
    candidates = {}
    for record in exact_records(query):
        add_candidate(candidates, record, f'{source_prefix}_exact')
    for record, score in fuzzy_candidate_records(query):
        add_candidate(candidates, record, f'{source_prefix}_fuzzy', score)
    return candidates


def merge_candidates(*candidate_sets):
    merged = {}
    for candidate_set in candidate_sets:
        for corp_code, candidate in candidate_set.items():
            record = candidate['record']
            for source in candidate['sources']:
                add_candidate(merged, record, source, candidate['fuzzy_score'])
    return merged


def needs_llm_expansion(raw_company_name):
    # 고유 exact는 OpenDART의 사실 검증만으로 충분하므로 LLM을 부르지 않는다.
    return len(exact_records(raw_company_name)) != 1

for case in TEST_CASES:
    case['LLM 호출 대상'] = needs_llm_expansion(case['원문 기업 표현'])

print('LLM 호출 대상 수:', sum(case['LLM 호출 대상'] for case in TEST_CASES))
print('LLM 미호출 대상 수:', sum(not case['LLM 호출 대상'] for case in TEST_CASES))

## 3. LangChain 구조화 출력 설정

스키마는 LLM이 `corp_code`, 투자 의견, 관련 계열사 이름을 임의로 반환하지 못하도록 검색어 목록만 허용한다. `include_raw=True`는 파싱 실패 여부와 모델 메타데이터를 함께 보존하기 위한 옵션이다.

`temperature=0`은 이번 후보의 실행 조건을 고정하기 위한 값이다. 동일 결과를 절대 보장하지는 않으며, 나중에 모델·프롬프트 비교 시 같은 값으로 통제한다.

In [ ]:
class EntitySearchPlan(BaseModel):
    decision: Literal['expand', 'unknown'] = Field(
        description='정식 기업명으로 해석할 검색어가 있으면 expand, 없으면 unknown'
    )
    search_terms: list[str] = Field(
        default_factory=list,
        max_length=2,
        description='OpenDART에서 다시 검색할 기업명 후보. 최대 2개, corp_code 금지.'
    )


llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=TEMPERATURE,
)
structured_llm = llm.with_structured_output(EntitySearchPlan, include_raw=True)

try:
    langchain_version = version('langchain')
    langchain_ollama_version = version('langchain-ollama')
except PackageNotFoundError as error:
    raise RuntimeError(f'필요한 패키지를 찾지 못했습니다: {error.name}') from error

RUN_CONFIG = {
    'run_at_utc': datetime.now(timezone.utc).isoformat(),
    'model': MODEL_NAME,
    'temperature': TEMPERATURE,
    'prompt_version': PROMPT_VERSION,
    'langchain_version': langchain_version,
    'langchain_ollama_version': langchain_ollama_version,
}
RUN_CONFIG

In [ ]:
SYSTEM_PROMPT = '''
당신은 한국 기업 리서치 서비스의 기업명 검색어 보조기다.
사용자 질문과 원문 기업 표현을 보고, 그 표현이 명백한 줄임말·오타·한글/영문 표기 차이일 때만 OpenDART 검색어 후보를 제안한다.

반드시 지킬 규칙:
1. 사용자가 언급하지 않은 계열사, 경쟁사, 관련 기업을 추가하지 않는다.
2. corp_code, 종목코드, 투자 의견, 설명 문장을 반환하지 않는다.
3. 확신이 없거나 포괄 표현이면 decision='unknown', search_terms=[]로 반환한다.
4. 제안이 필요한 경우에도 정식 기업명 검색어를 최대 2개만 반환한다.
5. 존재 여부를 추측하지 말고, 검색어 후보만 만든다. 실제 기업 검증은 OpenDART가 담당한다.
'''


def build_messages(question, raw_company_name):
    return [
        ('system', SYSTEM_PROMPT),
        ('human', f'사용자 질문: {question}\n원문 기업 표현: {raw_company_name}'),
    ]


def clean_search_terms(plan):
    if plan is None or plan.decision != 'expand':
        return [], []
    cleaned = []
    rejected = []
    seen = set()
    for term in plan.search_terms:
        term = term.strip()
        key = normalize_name(term)
        # corp_code·종목코드처럼 숫자로만 된 값은 기업명 검색어가 아니다.
        if not term or len(term) > 80 or not key or key in seen or key.isdigit():
            rejected.append(term)
            continue
        seen.add(key)
        cleaned.append(term)
    return cleaned[:2], rejected


# 단일 사례로 스키마·원문 응답·파싱 결과를 먼저 확인한다.
# 이 셀이 실패해도 다음 A/B 셀은 사례별 오류를 기록하며 계속 실행할 수 있다.
try:
    smoke_result = structured_llm.invoke(build_messages('하닉 실적을 정리해줘.', '하닉'))
    print('파싱 결과:', smoke_result['parsed'])
    print('파싱 오류:', smoke_result['parsing_error'])
    print('정리된 검색어:', clean_search_terms(smoke_result['parsed'])[0])
    print('응답 메타데이터:', smoke_result['raw'].response_metadata if smoke_result['raw'] else None)
except Exception as error:
    print(f'단일 호출 확인 실패: {type(error).__name__}: {error}')

## 4. A/B 실행

LLM은 비용과 시간이 드는 로컬 모델 호출이므로 각 입력당 한 번만 호출한다. 따라서 이 노트북의 p50/p95는 **입력들 사이의 1회 실행 분포**이며, 모델 성능의 통계적 반복 측정은 아니다.

LLM이 제안한 검색어는 OpenDART 인덱스에서 다시 탐색한다. LLM 제안만으로 `RESOLVED`를 만들지 않으며, 원문 exact가 복수거나 LLM 확장이 필요했던 사례는 이후 정책 평가 전까지 `NEEDS_CONFIRMATION` 후보로 유지한다.

In [ ]:
def candidate_view(candidate):
    record = candidate['record']
    return {
        '법인명': record['corp_name'],
        'corp_code': record['corp_code'],
        '종목코드': record['stock_code'].strip() or None,
        '출처': '+'.join(sorted(candidate['sources'])),
        '문자열 유사 점수': round(candidate['fuzzy_score'], 1) if candidate['fuzzy_score'] is not None else None,
    }


def safe_status(raw_company_name, merged_candidates, llm_called):
    raw_exact_count = len(exact_records(raw_company_name))
    if raw_exact_count == 1:
        return 'RESOLVED_RAW_EXACT'
    if raw_exact_count > 1:
        return 'NEEDS_CONFIRMATION'
    if llm_called or merged_candidates:
        return 'NEEDS_CONFIRMATION'
    return 'NOT_FOUND'


rows = []
for case in TEST_CASES:
    raw_name = case['원문 기업 표현']
    started_at = time.perf_counter()
    baseline_candidates = union_candidates(raw_name, source_prefix='raw')
    baseline_elapsed_ms = (time.perf_counter() - started_at) * 1_000

    llm_called = case['LLM 호출 대상']
    llm_elapsed_ms = None
    parsed = None
    parsing_error = None
    proposed_terms = []
    rejected_terms = []
    response_metadata = None
    expanded_candidates = baseline_candidates

    if llm_called:
        started_at = time.perf_counter()
        try:
            result = structured_llm.invoke(build_messages(case['질문'], raw_name))
            parsed = result['parsed']
            parsing_error = result['parsing_error']
            proposed_terms, rejected_terms = clean_search_terms(parsed)
            response_metadata = result['raw'].response_metadata if result['raw'] else None
            term_candidate_sets = [union_candidates(term, source_prefix='llm') for term in proposed_terms]
            expanded_candidates = merge_candidates(baseline_candidates, *term_candidate_sets)
        except Exception as error:
            # 모델·연결 오류도 한 사례의 실패로 기록하고 기준선 집계는 계속한다.
            parsing_error = f'{type(error).__name__}: {error}'
        finally:
            llm_elapsed_ms = (time.perf_counter() - started_at) * 1_000

    expected_codes = {target_records[key]['corp_code'] for key in case['기대 대상']}
    baseline_codes = set(baseline_candidates)
    expanded_codes = set(expanded_candidates)
    # fuzzy는 어떤 문자열에도 후보를 낼 수 있어 유효성 판단에 쓰지 않는다.
    term_validity = [bool(exact_records(term)) for term in proposed_terms]
    rows.append({
        **case,
        '기대 corp_code': sorted(expected_codes),
        '기준선 정답 포함': bool(expected_codes & baseline_codes) if expected_codes else None,
        '확장 후 정답 포함': bool(expected_codes & expanded_codes) if expected_codes else None,
        'LLM 신규 기여': bool((expected_codes & expanded_codes) - baseline_codes) if expected_codes else None,
        'LLM 신규 후보 주입': bool(expanded_codes - baseline_codes),
        'LLM 제안 검색어': proposed_terms,
        'LLM 검색어 유효 여부': term_validity,
        '거부된 LLM 검색어': rejected_terms,
        '구조화 출력 성공': None if not llm_called else parsing_error is None and parsed is not None,
        'LLM 호출 시간(ms)': llm_elapsed_ms,
        '기준선 후보 시간(ms)': baseline_elapsed_ms,
        '기준선 후보 수': len(baseline_candidates),
        '확장 후 후보 수': len(expanded_candidates),
        '안전 상태': safe_status(raw_name, expanded_candidates, llm_called),
        '기준선 후보': [candidate_view(item) for item in baseline_candidates.values()],
        '확장 후 후보': [candidate_view(item) for item in expanded_candidates.values()],
        'LLM 응답 메타데이터': response_metadata,
        '구조화 출력 오류': str(parsing_error) if parsing_error else None,
    })

len(rows)

In [ ]:
positive_rows = [row for row in rows if row['기대 corp_code']]
llm_rows = [row for row in rows if row['LLM 호출 대상']]
no_target_llm_rows = [row for row in llm_rows if row['유형'] == '대상 부재']
all_terms = [valid for row in llm_rows for valid in row['LLM 검색어 유효 여부']]
llm_latencies = [row['LLM 호출 시간(ms)'] for row in llm_rows if row['LLM 호출 시간(ms)'] is not None]

def percentile_95(values):
    if not values:
        return None
    return sorted(values)[ceil(len(values) * 0.95) - 1]

summary = {
    '평가 사례 수': len(rows),
    '정답 기업 사례 수': len(positive_rows),
    'A. 기준선 Candidate Recall': sum(row['기준선 정답 포함'] for row in positive_rows) / len(positive_rows),
    'B. LLM 확장 Candidate Recall': sum(row['확장 후 정답 포함'] for row in positive_rows) / len(positive_rows),
    'LLM Expansion Contribution': sum(row['LLM 신규 기여'] for row in positive_rows) / len(positive_rows),
    'Structured-output Success Rate': sum(row['구조화 출력 성공'] for row in llm_rows) / len(llm_rows),
    'LLM Search-term Validity': sum(all_terms) / len(all_terms) if all_terms else None,
    'LLM Call Rate': len(llm_rows) / len(rows),
    'No-target LLM Candidate Injection Rate': (
        sum(row['LLM 신규 후보 주입'] for row in no_target_llm_rows) / len(no_target_llm_rows)
        if no_target_llm_rows else None
    ),
    '숫자형 LLM 검색어 거부 수': sum(len(row['거부된 LLM 검색어']) for row in llm_rows),
    'A. 평균 후보 수': sum(row['기준선 후보 수'] for row in rows) / len(rows),
    'B. 평균 후보 수': sum(row['확장 후 후보 수'] for row in rows) / len(rows),
    'LLM 호출 p50(ms)': median(llm_latencies) if llm_latencies else None,
    'LLM 호출 p95(ms)': percentile_95(llm_latencies),
    '모델': MODEL_NAME,
    'temperature': TEMPERATURE,
}
summary

In [ ]:
import pandas as pd

result_columns = [
    '원문 기업 표현', '유형', '기대 상태', 'LLM 호출 대상', 'LLM 제안 검색어',
    '기준선 정답 포함', '확장 후 정답 포함', 'LLM 신규 기여',
    '구조화 출력 성공', 'LLM 검색어 유효 여부', '거부된 LLM 검색어', 'LLM 신규 후보 주입', '안전 상태',
    '기준선 후보 수', '확장 후 후보 수', 'LLM 호출 시간(ms)', '구조화 출력 오류',
]
display(pd.DataFrame(rows)[result_columns])

## 5. 결과 해석 기준

- B의 Candidate Recall이 A보다 높고, 그 차이가 실제 줄임말·오타 사례에서 나온다면 LLM 확장은 후보 누락을 줄이는 근거가 된다.
- Candidate Recall이 같고 LLM 신규 기여가 0에 가깝다면, 이 모델·프롬프트 조합은 비용과 지연시간 때문에 채택할 이유가 약하다.
- 대상 부재 입력에서 실제 OpenDART 후보를 만드는 제안이 많으면, 그럴듯한 실제 기업으로 잘못 이어질 위험이 있으므로 프롬프트 또는 호출 조건을 수정해야 한다.
- `삼전`처럼 원문 후보가 복수인 사례가 LLM 제안으로 하나의 `RESOLVED`가 되면 안 된다. 이번 셀의 `안전 상태`가 `NEEDS_CONFIRMATION`인지 확인한다.
- 이 결과만으로 최종 자동 해석 정책을 채택하지 않는다. 후보 확장이 유효하다고 확인된 경우에만 다음 평가에서 사용자 확인 정책과 안전한 자동 해석 범위를 별도로 비교한다.

실행 결과를 저장한 뒤 알려주면, 실제 출력·오류를 읽고 A/B 결과와 실패 사례를 해석한 후 HTML 결과 보고서 작성 여부를 함께 결정한다.